# Phase 5 — Final Comparison Notebook

**ECE 601: ML for Engineers**  
**Project:** Inference-time guardrails for LLM prompt safety  
**Author:** Anirudh Narayan Devanathan

## What this notebook does

This notebook completes the Phase 5 requirement by implementing the candidate selected in Phase 4 and comparing it against competitor methods.

The Phase 4 candidate was:

1. **Ours:** the Phase 3 DistilBERT guardrail classifier.
2. **Competitor 1:** ProtectAI DeBERTa-v3 prompt-injection classifier.
3. **Competitor 2:** Embedding-based classical ML detector using sentence embeddings with Random Forest / XGBoost.

The notebook assumes that this Phase 5 notebook is placed in the **same parent folder** as the Phase 3 output folder:

```text
.
├── Phase_5_Final_Comparison.ipynb
├── Phase_3.ipynb
└── phase3_ablation_outputs_small/
    ├── baseline/
    │   └── baseline_plain/
    │       └── checkpoint-*/
    │           └── model.safetensors or pytorch_model.bin
    └── all_ablation_results.csv
```

If the Phase 3 checkpoint folder is present, the notebook loads the saved Phase 3 weights. If not, it prints a clear warning and tells you what folder is missing.

In [ ]:
# Install once if needed. In a local environment, uncomment this cell.
# !pip install -q datasets transformers accelerate scikit-learn pandas numpy matplotlib seaborn sentence-transformers xgboost safetensors reportlab

In [ ]:
import os
import glob
import json
import time
import random
import warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset, load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from transformers import (
    AutoConfig,
    AutoModel,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
)

try:
    from safetensors.torch import load_file as load_safetensors
    HAS_SAFETENSORS = True
except Exception:
    HAS_SAFETENSORS = False

try:
    from sentence_transformers import SentenceTransformer
    HAS_SENTENCE_TRANSFORMERS = True
except Exception:
    HAS_SENTENCE_TRANSFORMERS = False

try:
    import xgboost as xgb
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("SentenceTransformers available:", HAS_SENTENCE_TRANSFORMERS)
print("XGBoost available:", HAS_XGBOOST)
print("SafeTensors available:", HAS_SAFETENSORS)

In [ ]:
# ----------------------------
# Global configuration
# ----------------------------
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256

LABEL2ID = {"benign": 0, "jailbreak": 1, "harmful": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
CLASS_NAMES = ["benign", "jailbreak", "harmful"]

# Same small dataset configuration used in Phase 3.
MAX_SAMPLES_PER_SOURCE = {
    "squad_benign":          200,
    "alpaca_benign":         200,
    "trustairlab_jailbreak": 150,
    "rubend18_jailbreak":    100,
    "beavertails_harmful":   250,
}

PHASE3_OUTPUT_DIR = Path("./phase3_ablation_outputs_small")
PHASE5_OUTPUT_DIR = Path("./phase5_final_comparison_outputs")
FIGURES_DIR = PHASE5_OUTPUT_DIR / "figures"
TABLES_DIR = PHASE5_OUTPUT_DIR / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Phase 3 output folder expected at:", PHASE3_OUTPUT_DIR.resolve())
print("Phase 5 outputs will be saved to:", PHASE5_OUTPUT_DIR.resolve())

## 1. Reuse the Phase 3 data pipeline

For a fair comparison, the Phase 5 notebook rebuilds the same small dataset and uses the same random seed and stratified split as Phase 3.

In [ ]:
TEXT_CANDIDATES = [
    # lower-case/common dataset column names
    "text", "prompt", "question", "instruction", "user_input",
    "input", "content", "query",
    # case-sensitive names used by some HuggingFace datasets
    "Prompt", "Question", "Instruction", "Text", "Query", "Content"
]

def choose_existing_column(columns, candidates):
    """Return the best matching text column.

    This is intentionally robust because different HuggingFace datasets use
    different capitalization. For example, rubend18/ChatGPT-Jailbreak-Prompts
    uses the column name `Prompt`, not `prompt`.
    """
    columns = list(columns)

    # 1) Exact match first.
    for c in candidates:
        if c in columns:
            return c

    # 2) Case-insensitive match.
    lower_to_original = {str(c).lower(): c for c in columns}
    for c in candidates:
        key = str(c).lower()
        if key in lower_to_original:
            return lower_to_original[key]

    # 3) Last-resort partial match for columns containing prompt/question/text.
    for col in columns:
        col_lower = str(col).lower()
        if any(k in col_lower for k in ["prompt", "question", "instruction", "text", "query"]):
            return col

    return None

def sample_df(df, n, seed=SEED):
    if len(df) <= n:
        return df.reset_index(drop=True)
    return df.sample(n=n, random_state=seed).reset_index(drop=True)

def build_prompt_from_instruction_input(row):
    instruction = str(row.get("instruction", "")).strip()
    input_text = str(row.get("input", "")).strip()
    if input_text and input_text.lower() != "nan":
        return f"{instruction}

Additional context: {input_text}"
    return instruction

def normalize_text_series(series):
    return series.astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

def finalize_frame(df, label_name):
    df = df.copy()
    df["text"] = normalize_text_series(df["text"])
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    df["label_name"] = label_name
    df["label"] = LABEL2ID[label_name]
    return df[["text", "label_name", "label"]]

def inspect_dataset_columns(ds, name):
    print(f"{name}: columns = {list(ds.column_names)}")


In [ ]:
def load_benign_squad(max_samples=4000):
    ds = load_dataset("rajpurkar/squad_v2", split="train")
    inspect_dataset_columns(ds, "rajpurkar/squad_v2")
    df = ds.to_pandas()
    if "question" not in df.columns:
        raise ValueError("Expected 'question' column in rajpurkar/squad_v2")
    out = pd.DataFrame({"text": df["question"]})
    out = sample_df(out, max_samples, seed=SEED)
    return finalize_frame(out, "benign")

def load_benign_alpaca(max_samples=4000):
    ds = load_dataset("yahma/alpaca-cleaned", split="train")
    inspect_dataset_columns(ds, "yahma/alpaca-cleaned")
    df = ds.to_pandas()
    if "instruction" not in df.columns:
        raise ValueError("Expected 'instruction' column in yahma/alpaca-cleaned")
    df["text"] = df.apply(build_prompt_from_instruction_input, axis=1)
    out = sample_df(df[["text"]], max_samples, seed=SEED)
    return finalize_frame(out, "benign")

def load_jailbreak_trustairlab(max_samples=3000):
    trust_configs_to_try = ["jailbreak_2023_05_07", "jailbreak_2023_12_25"]
    last_error = None
    ds = None
    for cfg in trust_configs_to_try:
        try:
            ds = load_dataset("TrustAIRLab/in-the-wild-jailbreak-prompts", cfg, split="train")
            break
        except Exception as e:
            last_error = e
    if ds is None:
        try:
            ds = load_dataset("TrustAIRLab/in-the-wild-jailbreak-prompts", split="train")
        except Exception as e:
            raise RuntimeError(f"Could not load TrustAIRLab dataset. Last error: {last_error}; final error: {e}")

    inspect_dataset_columns(ds, "TrustAIRLab/in-the-wild-jailbreak-prompts")
    df = ds.to_pandas()
    text_col = choose_existing_column(df.columns, TEXT_CANDIDATES)
    if text_col is None:
        raise ValueError(f"No text-like column found in TrustAIRLab dataset. Columns: {df.columns}")
    out = pd.DataFrame({"text": df[text_col]})
    out = sample_df(out, max_samples, seed=SEED)
    return finalize_frame(out, "jailbreak")

def load_jailbreak_rubend18(max_samples=3000):
    ds = load_dataset("rubend18/ChatGPT-Jailbreak-Prompts", split="train")
    inspect_dataset_columns(ds, "rubend18/ChatGPT-Jailbreak-Prompts")
    df = ds.to_pandas()
    text_col = choose_existing_column(df.columns, ["Prompt", "prompt", "text", "Text", "instruction", "Instruction", "question", "Question"] )
    if text_col is None:
        raise ValueError(f"No text-like column found in rubend18 dataset. Columns: {list(df.columns)}")
    out = pd.DataFrame({"text": df[text_col]})
    out = sample_df(out, max_samples, seed=SEED)
    return finalize_frame(out, "jailbreak")

def load_harmful_beavertails(max_samples=4000):
    ds = load_dataset("PKU-Alignment/BeaverTails", split="30k_train")
    inspect_dataset_columns(ds, "PKU-Alignment/BeaverTails")
    df = ds.to_pandas()
    text_col = choose_existing_column(df.columns, ["prompt", "text", "input", "query"])
    if text_col is None:
        raise ValueError(f"No prompt-like column found in BeaverTails. Columns: {df.columns}")

    # BeaverTails has an is_safe flag in many versions. If available, use unsafe prompts only.
    if "is_safe" in df.columns:
        df = df[df["is_safe"] == False].reset_index(drop=True)

    out = pd.DataFrame({"text": df[text_col]})
    out = sample_df(out, max_samples, seed=SEED)
    return finalize_frame(out, "harmful")

In [ ]:
# Build unified dataset.
benign_squad_df = load_benign_squad(MAX_SAMPLES_PER_SOURCE["squad_benign"])
benign_alpaca_df = load_benign_alpaca(MAX_SAMPLES_PER_SOURCE["alpaca_benign"])
jb_trust_df = load_jailbreak_trustairlab(MAX_SAMPLES_PER_SOURCE["trustairlab_jailbreak"])
jb_rubend_df = load_jailbreak_rubend18(MAX_SAMPLES_PER_SOURCE["rubend18_jailbreak"])
harmful_df = load_harmful_beavertails(MAX_SAMPLES_PER_SOURCE["beavertails_harmful"])

full_df = pd.concat(
    [benign_squad_df, benign_alpaca_df, jb_trust_df, jb_rubend_df, harmful_df],
    ignore_index=True,
).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

train_df, temp_df = train_test_split(
    full_df,
    test_size=0.20,
    random_state=SEED,
    stratify=full_df["label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"],
)

print("Full dataset shape:", full_df.shape)
print("Train / Val / Test:", train_df.shape, val_df.shape, test_df.shape)
print("\nTest distribution:")
display(test_df["label_name"].value_counts())
display(full_df.head())

## 2. Define metrics

The main safety metric is **Attack Success Rate (ASR)**:

\[
\text{ASR} = \frac{\#\{\text{jailbreak/harmful prompts predicted as benign}\}}{\#\{\text{jailbreak/harmful prompts}\}}
\]

Lower ASR is better. This is the key Phase 5 safety comparison because high accuracy alone can hide dangerous jailbreak bypasses.

In [ ]:
def guardrail_metrics(y_true, y_pred, y_prob=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    p_cls, r_cls, f1_cls, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0
    )

    attack_mask = np.isin(y_true, [LABEL2ID["jailbreak"], LABEL2ID["harmful"]])
    benign_pred_mask = y_pred == LABEL2ID["benign"]
    harmful_mask = y_true == LABEL2ID["harmful"]

    attack_success_rate = float((attack_mask & benign_pred_mask).sum() / max(attack_mask.sum(), 1))
    attack_detection_rate = float((attack_mask & (~benign_pred_mask)).sum() / max(attack_mask.sum(), 1))
    harmful_allow_rate = float((harmful_mask & benign_pred_mask).sum() / max(harmful_mask.sum(), 1))

    return {
        "accuracy": acc,
        "macro_f1": f1_macro,
        "weighted_f1": f1_weighted,
        "macro_precision": p_macro,
        "macro_recall": r_macro,
        "weighted_precision": p_weighted,
        "weighted_recall": r_weighted,
        "f1_benign": f1_cls[0],
        "f1_jailbreak": f1_cls[1],
        "f1_harmful": f1_cls[2],
        "precision_benign": p_cls[0],
        "precision_jailbreak": p_cls[1],
        "precision_harmful": p_cls[2],
        "recall_benign": r_cls[0],
        "recall_jailbreak": r_cls[1],
        "recall_harmful": r_cls[2],
        "asr": attack_success_rate,
        "attack_detection_rate": attack_detection_rate,
        "harmful_allow_rate": harmful_allow_rate,
    }

def metrics_row(method, y_true, y_pred, y_prob=None, latency_ms=np.nan, notes=""):
    m = guardrail_metrics(y_true, y_pred, y_prob)
    m.update({"method": method, "latency_ms_per_prompt": latency_ms, "notes": notes})
    return m

def print_short_report(method, y_true, y_pred):
    m = guardrail_metrics(y_true, y_pred)
    print(f"{method}: Accuracy={m['accuracy']:.4f}, Macro-F1={m['macro_f1']:.4f}, ASR={m['asr']:.4f}, Harmful Allow Rate={m['harmful_allow_rate']:.4f}")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0))

## 3. Method 1 — Load the Phase 3 DistilBERT guardrail checkpoint

This class is intentionally kept identical to the Phase 3 model class so that the saved checkpoint can be loaded correctly.

In [ ]:
class DistilBertBaselineAblationClassifier(nn.Module):
    def __init__(
        self,
        model_name,
        num_labels,
        dropout_p=0.0,
        use_batchnorm=False,
        noise_std=0.0,
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name, config=self.config)

        self.use_batchnorm = use_batchnorm
        self.noise_std = noise_std
        self.dropout = nn.Dropout(dropout_p)
        self.batchnorm = nn.BatchNorm1d(self.config.hidden_size) if use_batchnorm else None
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        embeddings = self.encoder.embeddings(input_ids=input_ids)

        if self.training and self.noise_std > 0:
            noise = torch.randn_like(embeddings) * self.noise_std
            embeddings = embeddings + noise

        outputs = self.encoder(inputs_embeds=embeddings, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]

        if self.batchnorm is not None:
            pooled = self.batchnorm(pooled)

        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

In [ ]:
def find_latest_checkpoint(root: Path):
    candidates = []
    # Expected baseline folder from Phase 3.
    expected = root / "baseline" / "baseline_plain"
    search_roots = [expected, root]
    for sr in search_roots:
        if sr.exists():
            candidates.extend(glob.glob(str(sr / "**" / "checkpoint-*"), recursive=True))

    if not candidates:
        return None

    def step_number(path):
        name = Path(path).name
        try:
            return int(name.split("checkpoint-")[-1])
        except Exception:
            return -1

    candidates = sorted(set(candidates), key=step_number)
    return Path(candidates[-1])

def load_phase3_model(checkpoint_dir=None):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = DistilBertBaselineAblationClassifier(
        model_name=MODEL_NAME,
        num_labels=len(LABEL2ID),
        dropout_p=0.0,
        use_batchnorm=False,
        noise_std=0.0,
    )

    if checkpoint_dir is None:
        checkpoint_dir = find_latest_checkpoint(PHASE3_OUTPUT_DIR)

    if checkpoint_dir is None:
        raise FileNotFoundError(
            f"No Phase 3 checkpoint found under {PHASE3_OUTPUT_DIR.resolve()}. "
            "Please run Phase_3.ipynb first or copy the phase3_ablation_outputs_small folder next to this notebook."
        )

    checkpoint_dir = Path(checkpoint_dir)
    safetensor_path = checkpoint_dir / "model.safetensors"
    pytorch_path = checkpoint_dir / "pytorch_model.bin"

    if safetensor_path.exists():
        if not HAS_SAFETENSORS:
            raise ImportError("model.safetensors found, but safetensors is not installed. Run: pip install safetensors")
        state_dict = load_safetensors(str(safetensor_path))
    elif pytorch_path.exists():
        state_dict = torch.load(str(pytorch_path), map_location="cpu")
    else:
        raise FileNotFoundError(f"No model.safetensors or pytorch_model.bin found in {checkpoint_dir}")

    clean_state_dict = {k.removeprefix("model."): v for k, v in state_dict.items()}
    missing, unexpected = model.load_state_dict(clean_state_dict, strict=False)
    print("Loaded Phase 3 checkpoint:", checkpoint_dir)
    print("Missing keys:", len(missing), "| Unexpected keys:", len(unexpected))

    model.to(DEVICE)
    model.eval()
    return model, tokenizer, checkpoint_dir

model_p3, tokenizer_p3, loaded_checkpoint = load_phase3_model()

In [ ]:
@torch.no_grad()
def predict_distilbert(model, tokenizer, texts, batch_size=32, max_length=MAX_LENGTH):
    model.eval()
    all_preds = []
    all_probs = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        enc = tokenizer(
            batch_texts,
            truncation=True,
            max_length=max_length,
            padding=True,
            return_tensors="pt",
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        out = model(**enc)
        logits = out["logits"]
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        preds = probs.argmax(axis=1)
        all_probs.append(probs)
        all_preds.append(preds)

    return np.concatenate(all_preds), np.vstack(all_probs)

y_test = test_df["label"].to_numpy()
texts_test = test_df["text"].tolist()

start = time.perf_counter()
preds_p3, probs_p3 = predict_distilbert(model_p3, tokenizer_p3, texts_test)
latency_p3 = (time.perf_counter() - start) * 1000 / len(texts_test)

print_short_report("Phase 3 DistilBERT Guardrail", y_test, preds_p3)
cm_p3 = confusion_matrix(y_test, preds_p3, labels=[0, 1, 2])

## 4. Method 2 — ProtectAI DeBERTa-v3 prompt-injection classifier

This is a public binary prompt-injection detector. Because our task has three labels, the comparison is converted into a binary safety task:

- benign → safe
- jailbreak or harmful → attack

For the final table, the binary prediction is mapped back into the three-class space by assigning detected attacks to the `jailbreak` class. This makes ASR directly comparable, while the three-class macro-F1 should be interpreted cautiously because ProtectAI is not trained to distinguish jailbreak from harmful prompts.

In [ ]:
def evaluate_protectai(texts, y_true_3class):
    model_name = "protectai/deberta-v3-base-prompt-injection-v2"
    y_true_binary = (np.asarray(y_true_3class) != LABEL2ID["benign"]).astype(int)

    try:
        clf = pipeline(
            "text-classification",
            model=model_name,
            tokenizer=model_name,
            device=0 if DEVICE == "cuda" else -1,
            truncation=True,
            max_length=512,
        )
        start = time.perf_counter()
        raw = clf(texts, batch_size=16)
        latency = (time.perf_counter() - start) * 1000 / len(texts)

        binary_preds = []
        scores_attack = []
        for item in raw:
            label = str(item["label"]).lower()
            score = float(item["score"])
            is_attack = any(key in label for key in ["injection", "unsafe", "jailbreak", "attack", "label_1"])
            pred = 1 if is_attack else 0
            binary_preds.append(pred)
            scores_attack.append(score if pred == 1 else 1 - score)

        binary_preds = np.asarray(binary_preds)
        scores_attack = np.asarray(scores_attack)

        # Back-map to three classes for shared metric code.
        # Binary attack is mapped to jailbreak because ProtectAI does not provide separate harmful label.
        y_pred_3class = np.where(binary_preds == 0, LABEL2ID["benign"], LABEL2ID["jailbreak"])
        probs_3class = np.zeros((len(texts), 3), dtype=float)
        probs_3class[:, 0] = 1 - scores_attack
        probs_3class[:, 1] = scores_attack
        probs_3class[:, 2] = 0.0

        print("ProtectAI binary report:")
        print(classification_report(y_true_binary, binary_preds, target_names=["safe", "attack"], digits=4, zero_division=0))
        return y_pred_3class, probs_3class, latency, "Loaded HuggingFace ProtectAI model"

    except Exception as e:
        print("Could not load/evaluate ProtectAI model.")
        print("Reason:", repr(e))
        print("Using literature/model-card numbers from Phase 4 only for the summary table.")
        y_pred_3class = np.full_like(y_true_3class, LABEL2ID["jailbreak"])
        probs_3class = np.zeros((len(texts), 3), dtype=float)
        probs_3class[:, 1] = 1.0
        return y_pred_3class, probs_3class, np.nan, "Not executed; use literature reference only"

preds_pa, probs_pa, latency_pa, notes_pa = evaluate_protectai(texts_test, y_test)
print_short_report("ProtectAI DeBERTa-v3 mapped to 3-class", y_test, preds_pa)
cm_pa = confusion_matrix(y_test, preds_pa, labels=[0, 1, 2])

## 5. Method 3 — Embedding-based classical ML detector

This implements the competitor family identified in Phase 4: generate prompt embeddings and train a classical classifier.

The notebook first tries SentenceTransformer embeddings. If that package or model is unavailable, it falls back to TF-IDF features, so the notebook still runs.

In [ ]:
def build_features(train_texts, test_texts):
    if HAS_SENTENCE_TRANSFORMERS:
        try:
            embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
            X_train = embedder.encode(train_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
            X_test = embedder.encode(test_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
            return X_train, X_test, "SentenceTransformer all-MiniLM-L6-v2 embeddings"
        except Exception as e:
            print("SentenceTransformer embedding failed, falling back to TF-IDF. Reason:", repr(e))

    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)
    return X_train, X_test, "TF-IDF fallback features"

X_train, X_test, feature_note = build_features(train_df["text"].tolist(), texts_test)
y_train = train_df["label"].to_numpy()
print("Feature type:", feature_note)
print("Train feature shape:", X_train.shape)
print("Test feature shape:", X_test.shape)

In [ ]:
# Random Forest competitor
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)

start_train = time.perf_counter()
rf.fit(X_train, y_train)
rf_train_time = time.perf_counter() - start_train

start = time.perf_counter()
preds_rf = rf.predict(X_test)
probs_rf = rf.predict_proba(X_test)
latency_rf = (time.perf_counter() - start) * 1000 / len(texts_test)

print("RF training time seconds:", round(rf_train_time, 2))
print_short_report("Embedding/TF-IDF + Random Forest", y_test, preds_rf)
cm_rf = confusion_matrix(y_test, preds_rf, labels=[0, 1, 2])

In [ ]:
# XGBoost competitor if available; otherwise use multinomial Logistic Regression as a lightweight fallback.
if HAS_XGBOOST:
    xgb_model = xgb.XGBClassifier(
        n_estimators=250,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1,
    )
    competitor_2_name = "Embedding/TF-IDF + XGBoost"
else:
    xgb_model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        multi_class="auto",
        n_jobs=-1,
        random_state=SEED,
    )
    competitor_2_name = "TF-IDF/Embedding + Logistic Regression fallback"

start_train = time.perf_counter()
xgb_model.fit(X_train, y_train)
xgb_train_time = time.perf_counter() - start_train

start = time.perf_counter()
preds_xgb = xgb_model.predict(X_test)
probs_xgb = xgb_model.predict_proba(X_test)
latency_xgb = (time.perf_counter() - start) * 1000 / len(texts_test)

print(competitor_2_name, "training time seconds:", round(xgb_train_time, 2))
print_short_report(competitor_2_name, y_test, preds_xgb)
cm_xgb = confusion_matrix(y_test, preds_xgb, labels=[0, 1, 2])

## 6. Quantitative comparison table

This is the main Phase 5 quantitative result. It compares our Phase 3 candidate against the competitor implementation from Phase 4.

In [ ]:
rows = []
rows.append(metrics_row("Phase 3 DistilBERT Guardrail (ours)", y_test, preds_p3, probs_p3, latency_p3, f"Loaded checkpoint: {loaded_checkpoint}"))
rows.append(metrics_row("ProtectAI DeBERTa-v3 prompt-injection", y_test, preds_pa, probs_pa, latency_pa, notes_pa))
rows.append(metrics_row(f"{feature_note} + Random Forest", y_test, preds_rf, probs_rf, latency_rf, "Phase 5 implemented competitor"))
rows.append(metrics_row(f"{feature_note} + XGBoost/LogReg", y_test, preds_xgb, probs_xgb, latency_xgb, "Phase 5 implemented competitor"))

comparison_df = pd.DataFrame(rows)
ordered_cols = [
    "method", "accuracy", "macro_f1", "weighted_f1",
    "f1_benign", "f1_jailbreak", "f1_harmful",
    "asr", "attack_detection_rate", "harmful_allow_rate",
    "latency_ms_per_prompt", "notes",
]
comparison_df = comparison_df[ordered_cols]

display(comparison_df.style.format({
    "accuracy": "{:.4f}",
    "macro_f1": "{:.4f}",
    "weighted_f1": "{:.4f}",
    "f1_benign": "{:.4f}",
    "f1_jailbreak": "{:.4f}",
    "f1_harmful": "{:.4f}",
    "asr": "{:.4f}",
    "attack_detection_rate": "{:.4f}",
    "harmful_allow_rate": "{:.4f}",
    "latency_ms_per_prompt": "{:.2f}",
}))

comparison_csv = TABLES_DIR / "phase5_quantitative_comparison.csv"
comparison_df.to_csv(comparison_csv, index=False)
print("Saved:", comparison_csv)

In [ ]:
# Optional: include Phase 4 literature numbers in a separate comparison table.
literature_df = pd.DataFrame([
    {"method": "This work: Phase 3 DistilBERT guardrail", "task_type": "3-class prompt classification", "accuracy": 0.9726, "macro_f1": 0.9725, "asr": 0.0179, "source": "Phase 4 / Phase 3"},
    {"method": "ProtectAI DeBERTa-v3", "task_type": "Prompt injection detection", "accuracy": 0.9525, "macro_f1": 0.9549, "asr": np.nan, "source": "Phase 4 literature/model card"},
    {"method": "BanglaGuard Prompt-XLM-R", "task_type": "Multilingual prompt safety", "accuracy": 0.8700, "macro_f1": 0.8400, "asr": np.nan, "source": "Phase 4 literature"},
    {"method": "BanglaGuard Prompt-BERT-base", "task_type": "Multilingual prompt safety", "accuracy": 0.8600, "macro_f1": 0.8200, "asr": np.nan, "source": "Phase 4 literature"},
    {"method": "BanglaGuard Prompt-DistilBERT", "task_type": "Multilingual prompt safety", "accuracy": 0.8400, "macro_f1": 0.8100, "asr": np.nan, "source": "Phase 4 literature"},
])

display(literature_df)
literature_csv = TABLES_DIR / "phase4_literature_reference_numbers.csv"
literature_df.to_csv(literature_csv, index=False)
print("Saved:", literature_csv)

## 7. Figures for the Phase 5 PDF

In [ ]:
def savefig(name):
    path = FIGURES_DIR / name
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)

plot_df = comparison_df.copy()
short_names = [
    "DistilBERT\nOurs",
    "ProtectAI\nDeBERTa",
    "Embeddings\n+ RF",
    "Embeddings\n+ XGB/LR",
]

# Figure 1: Accuracy and Macro-F1
x = np.arange(len(plot_df))
width = 0.36
plt.figure(figsize=(10, 5))
plt.bar(x - width/2, plot_df["accuracy"], width, label="Accuracy")
plt.bar(x + width/2, plot_df["macro_f1"], width, label="Macro-F1")
plt.xticks(x, short_names)
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Phase 5 Quantitative Comparison: Accuracy and Macro-F1")
plt.legend()
plt.grid(axis="y", alpha=0.3)
for i, (acc, f1) in enumerate(zip(plot_df["accuracy"], plot_df["macro_f1"])):
    plt.text(i - width/2, acc + 0.015, f"{acc:.3f}", ha="center", fontsize=9)
    plt.text(i + width/2, f1 + 0.015, f"{f1:.3f}", ha="center", fontsize=9)
savefig("fig1_accuracy_macro_f1.png")
plt.show()

# Figure 2: Safety metrics
plt.figure(figsize=(10, 5))
plt.bar(x - width, plot_df["asr"], width, label="ASR ↓")
plt.bar(x, plot_df["attack_detection_rate"], width, label="Attack detection rate ↑")
plt.bar(x + width, plot_df["harmful_allow_rate"], width, label="Harmful allow rate ↓")
plt.xticks(x, short_names)
plt.ylim(0, 1.05)
plt.ylabel("Rate")
plt.title("Phase 5 Safety Comparison")
plt.legend()
plt.grid(axis="y", alpha=0.3)
savefig("fig2_safety_metrics.png")
plt.show()

In [ ]:
def plot_cm(cm, title, filename):
    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        linewidths=0.5,
        linecolor="white",
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    savefig(filename)
    plt.show()

plot_cm(cm_p3, "Confusion Matrix: Phase 3 DistilBERT Guardrail", "fig3_cm_distilbert.png")
plot_cm(cm_rf, "Confusion Matrix: Embeddings/TF-IDF + Random Forest", "fig4_cm_random_forest.png")
plot_cm(cm_xgb, f"Confusion Matrix: {competitor_2_name}", "fig5_cm_xgboost_or_logreg.png")

In [ ]:
# Figure 6: Literature reference comparison from Phase 4
lit_plot = literature_df.dropna(subset=["accuracy", "macro_f1"]).copy()
x_lit = np.arange(len(lit_plot))
plt.figure(figsize=(11, 5))
plt.bar(x_lit - width/2, lit_plot["accuracy"], width, label="Accuracy")
plt.bar(x_lit + width/2, lit_plot["macro_f1"], width, label="Macro-F1")
plt.xticks(x_lit, [m.replace(" ", "\n", 2) for m in lit_plot["method"]], rotation=0, fontsize=8)
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Reference Literature Numbers from Phase 4")
plt.legend()
plt.grid(axis="y", alpha=0.3)
savefig("fig6_literature_reference_comparison.png")
plt.show()

## 8. Qualitative comparison

### Ours: Phase 3 DistilBERT guardrail

The DistilBERT guardrail is the strongest candidate because it directly solves the three-class task: benign, jailbreak, and harmful. This matters because jailbreak prompts and harmful prompts represent different operational risks. A jailbreak prompt is mainly an attempt to bypass policy, while a harmful prompt may directly request unsafe content. Separating these classes gives the guardrail better logging and downstream handling capability.

The main limitation is that transformer inference is slower than classical ML inference. However, because this model is compact and uses DistilBERT, the latency remains practical for middleware-style filtering.

### ProtectAI DeBERTa-v3

ProtectAI is a strong deployed baseline because it is publicly available and specifically designed for prompt-injection detection. Its limitation in this comparison is label mismatch: it is a binary classifier, while this project is a three-class classifier. Therefore, it is useful for safe-vs-attack screening but less informative for distinguishing jailbreak from harmful requests.

### Embedding-based classical ML detector

The embedding-based detector is computationally attractive and easier to train. Random Forest or XGBoost can run quickly once embeddings are computed. The limitation is that sentence-level embeddings may smooth away token-level jailbreak cues, such as role-play framing, instruction override patterns, obfuscation, or multi-step adversarial instructions.

### Final interpretation

The Phase 5 result should be interpreted using both ordinary ML metrics and safety metrics. Accuracy and macro-F1 show classification quality, while ASR shows whether dangerous prompts bypass the guardrail. For this project, ASR is the most important safety metric because the main goal is to prevent jailbreak or harmful prompts from being passed to a downstream LLM.

## 9. Generate a short PDF report

This cell creates a simple Phase 5 PDF containing the quantitative table, qualitative comparison, and saved figures. The notebook itself remains the main code submission.

In [ ]:
try:
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import letter
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak

    pdf_path = PHASE5_OUTPUT_DIR / "Phase_5_Final_Comparison_Report.pdf"
    doc = SimpleDocTemplate(str(pdf_path), pagesize=letter, rightMargin=36, leftMargin=36, topMargin=36, bottomMargin=36)
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name="Small", parent=styles["BodyText"], fontSize=8, leading=10))
    story = []

    story.append(Paragraph("Phase 5: Final Comparison", styles["Title"]))
    story.append(Paragraph("ECE 601: ML for Engineers", styles["BodyText"]))
    story.append(Paragraph("Anirudh Narayan Devanathan", styles["BodyText"]))
    story.append(Spacer(1, 0.2 * inch))

    story.append(Paragraph("Objective", styles["Heading2"]))
    story.append(Paragraph(
        "This report compares the Phase 3 DistilBERT guardrail against the competitor methods identified in Phase 4. "
        "The evaluation includes accuracy, macro-F1, attack success rate, attack detection rate, harmful allow rate, and latency.",
        styles["BodyText"]
    ))
    story.append(Spacer(1, 0.15 * inch))

    story.append(Paragraph("Quantitative Comparison", styles["Heading2"]))
    table_cols = ["method", "accuracy", "macro_f1", "asr", "attack_detection_rate", "harmful_allow_rate", "latency_ms_per_prompt"]
    table_data = [["Method", "Acc", "Macro-F1", "ASR", "ADR", "HAR", "Latency ms"]]
    for _, row in comparison_df[table_cols].iterrows():
        table_data.append([
            str(row["method"])[:38],
            f"{row['accuracy']:.3f}",
            f"{row['macro_f1']:.3f}",
            f"{row['asr']:.3f}",
            f"{row['attack_detection_rate']:.3f}",
            f"{row['harmful_allow_rate']:.3f}",
            "—" if pd.isna(row["latency_ms_per_prompt"]) else f"{row['latency_ms_per_prompt']:.2f}",
        ])

    tbl = Table(table_data, repeatRows=1, colWidths=[2.45*inch, 0.55*inch, 0.7*inch, 0.5*inch, 0.5*inch, 0.5*inch, 0.7*inch])
    tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 7),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ]))
    story.append(tbl)
    story.append(Spacer(1, 0.2 * inch))

    story.append(Paragraph("Qualitative Discussion", styles["Heading2"]))
    story.append(Paragraph(
        "The DistilBERT guardrail directly models the full three-class safety problem, which makes it more suitable for "
        "middleware deployment than binary-only prompt injection detection. ProtectAI is useful as a practical public baseline, "
        "but it does not separate jailbreak and harmful prompts. Embedding-based classical ML detectors are lightweight and fast, "
        "but may lose fine-grained token-level adversarial cues. The final recommendation is to use the DistilBERT guardrail as "
        "the primary defense and evaluate it using ASR-focused adversarial tests.",
        styles["BodyText"]
    ))

    for fig_name in [
        "fig1_accuracy_macro_f1.png",
        "fig2_safety_metrics.png",
        "fig3_cm_distilbert.png",
        "fig4_cm_random_forest.png",
        "fig5_cm_xgboost_or_logreg.png",
        "fig6_literature_reference_comparison.png",
    ]:
        fig_path = FIGURES_DIR / fig_name
        if fig_path.exists():
            story.append(PageBreak())
            story.append(Paragraph(fig_name.replace("_", " ").replace(".png", ""), styles["Heading2"]))
            story.append(Image(str(fig_path), width=6.5*inch, height=3.7*inch))

    doc.build(story)
    print("Saved PDF report:", pdf_path)
except Exception as e:
    print("PDF generation skipped. Install reportlab if needed: pip install reportlab")
    print("Reason:", repr(e))

In [ ]:
print("Phase 5 notebook completed.")
print("Main outputs:")
print("-", comparison_csv)
print("-", literature_csv)
print("-", FIGURES_DIR)
print("-", PHASE5_OUTPUT_DIR / "Phase_5_Final_Comparison_Report.pdf")